In [143]:
import pandas as pd
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
gender_submission = pd.read_csv('gender_submission.csv')

In [144]:
train_df[train_df['Embarked'].isnull()]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
61,62,1,1,"Icard, Miss. Amelie",female,38.0,0,0,113572,80.0,B28,NaN
829,830,1,1,"Stone, Mrs. George Nelson (Martha Evelyn)",female,62.0,0,0,113572,80.0,B28,NaN


In [145]:
import pandas as pd
import numpy as np

def preprocess_titanic_data(df, train_pclass1_median=None, train_age_medians=None):
    """
    リークを完璧に排除した、TrainとTest共通の前処理関数
    """
    target_df = df.copy()
    
    # ① 敬称（Title）の切り出し
    target_df['Title'] = target_df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
    
    # ----------------------------------------------------
    # ② 年齢（Age）の中央値穴埋め（★鈴木さんの指摘通りに修正！）
    # ----------------------------------------------------
    if train_age_medians is not None:
        # trainから計算された「敬称ごとの中央値辞書」を使って、一人ずつ綺麗にマッピングして埋める
        # 例：AgeがNullでTitleがMasterの人には、trainのMaster中央値(3.5才など)が代入される
        target_df['Age'] = target_df['Age'].fillna(target_df['Title'].map(train_age_medians))
    
    # 最終保険（これも train の全体中央値を使うべきなので、後ほど外から渡せるようにしてもOKです）
    target_df['Age'] = target_df['Age'].fillna(target_df['Age'].median()) 
    
    # 10歳刻み化 ＆ 元のAgeをドロップ
    bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 120]
    labels = ['0s', '10s', '20s', '30s', '40s', '50s', '60s', '70s', '80s+']
    target_df['Age_Group'] = pd.cut(target_df['Age'], bins=bins, labels=labels, right=False)
    target_df = target_df.drop(columns=['Age'])
    
    # ③ デッキ予測（trainの中央値基準）
    target_df['Predicted_Deck'] = target_df['Cabin'].str[0]
    if train_pclass1_median is not None:
        target_df.loc[(target_df['Predicted_Deck'].isnull()) & (target_df['Pclass'] == 1) & (target_df['Fare'] >= train_pclass1_median), 'Predicted_Deck'] = 'C'
        target_df.loc[(target_df['Predicted_Deck'].isnull()) & (target_df['Pclass'] == 1) & (target_df['Fare'] < train_pclass1_median), 'Predicted_Deck'] = 'D'
    target_df.loc[(target_df['Predicted_Deck'].isnull()) & (target_df['Pclass'] == 2), 'Predicted_Deck'] = 'E'
    target_df.loc[(target_df['Predicted_Deck'].isnull()) & (target_df['Pclass'] == 3), 'Predicted_Deck'] = 'F'
    target_df['Predicted_Deck'] = target_df['Predicted_Deck'].fillna('F')
    
    # ④ VIP優待フラグ
    target_df['Is_VIP_Invitation'] = ((target_df['Pclass'] == 1) & (target_df['Fare'] == 0)).astype(int)
    
    # ⑤ 家族人数
    target_df['FamilySize'] = target_df['SibSp'] + target_df['Parch'] + 1
    # 1. まず、鈴木さんの推理通り「1等客でリッチなNull」をピンポイントでC港に
    target_df.loc[(target_df['Embarked'].isnull()) & (target_df['Pclass'] == 1) & (target_df['Fare'] > 50), 'Embarked'] = 'C'
    # 2. 【最終保険】もしこれ以外の想定外のNullが残っていたら、一番多い 'S'（または 'Unknown'）で埋める
    target_df['Embarked'] = target_df['Embarked'].fillna('S')
    # 1. 同じチケット番号の人が何人いるかを数えて、新しい列「Ticket_Count」を作る
    ticket_counts = target_df['Ticket'].value_counts()
    target_df['Ticket_Count'] = target_df['Ticket'].map(ticket_counts)
    # 2. 運賃をその人数で割り算して「1人あたりの運賃（Individual_Fare）」を作る
    target_df['Individual_Fare'] = target_df['Fare'] / target_df['Ticket_Count']
    # 3. 元の「Fare」は消す
    target_df = target_df.drop(columns=['Ticket_Count'])
    # 珍しい敬称を、一般的なもの（または 'Rare'）に置き換える辞書だが、AIに聞かないと無理だな
    title_mapping = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs', # フランス語の敬称などを統合
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare', 'Mlle': 'Miss',
    'Capt': 'Rare', 'Sir': 'Rare', 'Lady': 'Rare', 'Lady': 'Rare', 'the Countess': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare'
    }
    target_df['Title'] = target_df['Title'].map(title_mapping)
    # 万が一、辞書にない敬称があった場合の保険
    target_df['Title'] = target_df['Title'].fillna('Rare')
    return target_df

   

In [146]:
# --- 準備：すべての基準（ものさし）を【必ずtrainから】計算して保存する ---

# 1. 1等客の運賃中央値
train_pclass1_median = train_df[train_df['Pclass'] == 1]['Fare'].median()

# 2. 敬称ごとの年齢中央値（辞書形式で取得されます：{'Mr': 30, 'Miss': 21, ...}）
# ★一度仮でTitleを作ってから計算します
temp_train_title = train_df['Name'].str.split(',').str[1].str.split('.').str[0].str.strip()
train_age_medians = train_df.groupby(temp_train_title)['Age'].median().to_dict()


# --- 本番：同じものさしを両方に適用する ---

# ① train_df に適用
clean_train_df = preprocess_titanic_data(train_df, train_pclass1_median, train_age_medians)

# ② test_df に適用（★年齢の基準もtrainのものが使い回されるので、100%安全！）
clean_test_df = preprocess_titanic_data(test_df, train_pclass1_median, train_age_medians)

In [147]:
# 削除したい「ゴミ箱リスト」を作る
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']

# 訓練データから特徴量（X）と、予測対象の答え（y：生存フラグ）を分ける
X_train = clean_train_df.drop(columns=drop_cols + ['Survived'])
y_train = clean_train_df['Survived']

# テストデータからも同じ列を消す（答えの列はないので除外）
X_test = clean_test_df.drop(columns=drop_cols)

In [148]:
X_train.isnull().sum()

Pclass               0
Sex                  0
SibSp                0
Parch                0
Fare                 0
Embarked             0
Title                0
Age_Group            0
Predicted_Deck       0
Is_VIP_Invitation    0
FamilySize           0
Individual_Fare      0
dtype: int64

In [149]:
# こちらの方が実務でよく見る「文字列の列名リスト」を自動で作る技です
obj_cols = X_train.select_dtypes(include=['object']).columns

X_train[obj_cols] = X_train[obj_cols].astype('category')
X_test[obj_cols]  = X_test[obj_cols].astype('category')

In [150]:
# 分類用の LightGBM をインポートする
from lightgbm import LGBMClassifier

# モデルの作成（まずは標準的な設定でOKです！）
model = LGBMClassifier(
    random_state=42,
    n_estimators=100,
    learning_rate=0.05,
    )
# 🌟 いざ学習！磨き上げた特徴量（X）と、答え（y）をAIに食べさせます
model.fit(X_train, y_train)

print("学習が正常に完了しました！")

[LightGBM] [Info] Number of positive: 342, number of negative: 549
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000087 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 319
[LightGBM] [Info] Number of data points in the train set: 891, number of used features: 11
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.383838 -> initscore=-0.473288
[LightGBM] [Info] Start training from score -0.473288
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf

In [151]:
# 特徴量の重要度（Feature Importance）を綺麗に表示する
import pandas as pd

importance = pd.DataFrame({
    '特徴量': X_train.columns,
    '重要度': model.feature_importances_
}).sort_values(by='重要度', ascending=False)

print(importance)

                  特徴量   重要度
4                Fare  1087
11    Individual_Fare   797
7           Age_Group   500
5            Embarked   144
10         FamilySize   101
1                 Sex    91
2               SibSp    73
0              Pclass    70
3               Parch    63
6               Title    41
8      Predicted_Deck    22
9   Is_VIP_Invitation     0


In [152]:
# 1. テストデータ（X_test）の生存予測をする（1 または 0 が返ってきます）
y_pred = model.predict(X_test)

# 2. Kaggleなどの提出形式（指定のファイル形式）に合わせてデータフレームを作る
# ※最初にとっておいた、または test_df に入っている「PassengerId」とペアにします
submission = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': y_pred
})

# 3. CSVファイルとして保存する（インデックス番号は不要なので index=False にします）
submission.to_csv('submission_titanic.csv', index=False)

print("提出用ファイル 'submission_titanic.csv' が無事に作成されました！")

提出用ファイル 'submission_titanic.csv' が無事に作成されました！


In [153]:
# 作成した中身をサクッと確認
print(submission.head())

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         0
